![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 46 -- Competition: Shelter Cutouts

An animal shelter puts a photo of every cat and dog it cares for on its website. Each photo should show the animal alone, cut out from its background. Your goal is to mark, in every photo, **which pixels belong to the animal**.

**The photos.** 20 photos come with their masks (`train_images/` and `train_masks/`). A mask is a black-and-white picture of the same size as its photo: white (255) where the animal is, black (0) everywhere else. 1,000 photos come without masks (`test_images/`): you make their masks. The 37 breeds the photos show are listed below; not every breed is among the 20 labelled photos.

**The one rule of the task.** The only pretrained model you may use is the CLIP model below (`openai/clip-vit-base-patch16`), with any text you like for it. No other pretrained model and no other data may be used. Everything else is allowed, including training on the 20 labelled photos.

**Score.** For every test photo, the intersection over union (IoU) of your mask and the true mask: the pixels marked in both, divided by the pixels marked in either. Your score is the mean over the photos: 1 is perfect. Kaggle shows the score on 40% of the photos during the competition and ranks you at the end on the other 60%.

**Your file.** `submission.csv` with the columns `id` and `mask`: one row for every test photo, `id` the number of its file name, `mask` its mask as a run-length code (pairs "start length" of animal pixels, pixels numbered from 1, row by row). The function `generate_submit` below writes it from your masks.

**Dataset:** The setup cell leaves four things in `DATA_DIR`: `train_images/` (20 photos, `0.jpg` to `19.jpg`), `train_masks/` (their masks, `0.png` to `19.png`), `test_images/` (1,000 photos, `0.jpg` to `999.jpg`) and `sample_submission.csv` (a valid file that marks every pixel as animal). Photos of the Oxford-IIIT Pet dataset (CC BY-SA 4.0).

The clock started at the opening time, so start at once.

![A labelled photo and its mask](https://raw.githubusercontent.com/KAUST-Academy/Olympiad2027/main/L1/Comps/November_Camp/Day46_Shelter_Cutouts/sample.png)

A labelled photo and its mask: the animal is white, everything else black.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab transformers
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="cv-comp", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
from torch.nn import functional as F
from transformers import AutoProcessor, CLIPModel

np.random.seed(0)
torch.manual_seed(0)
print("torch", torch.__version__)


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/shelter-cutouts")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The breeds

The breeds of the cats and dogs in the photos. Not every breed is among the 20 labelled photos.

In [ ]:
class_names = ['american_bulldog', 'basset_hound', 'keeshond', 'British_Shorthair', 'Sphynx', 'pomeranian',
               'Egyptian_Mau', 'Birman', 'american_pit_bull_terrier', 'japanese_chin', 'Maine_Coon', 'beagle',
               'Bombay', 'wheaten_terrier', 'shiba_inu', 'havanese', 'miniature_pinscher', 'yorkshire_terrier',
               'boxer', 'scottish_terrier', 'newfoundland', 'chihuahua', 'saint_bernard', 'Persian', 'Bengal',
               'german_shorthaired', 'english_cocker_spaniel', 'leonberger', 'Siamese', 'Abyssinian',
               'staffordshire_bull_terrier', 'Ragdoll', 'pug', 'Russian_Blue', 'samoyed', 'english_setter',
               'great_pyrenees']
print(len(class_names), "breeds")


---
# The model

The only pretrained model this task allows.

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
model_name = "openai/clip-vit-base-patch16"

processor = AutoProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()
print("device:", device)


---
# Baseline

The starter solution, for each photo:

1. compute the CLIP embeddings of all breed names;
2. resize the photo to 224 x 224 (the size CLIP takes);
3. compute the CLIP embedding of the photo;
4. find the breed whose embedding is most similar to the photo's (zero-shot classification);
5. cut the photo into patches of 16 x 16 pixels (14 x 14 patches);
6. enlarge each patch to 224 x 224 and compute its CLIP embedding as a picture of its own;
7. take the similarity of every patch with the chosen breed: a 14 x 14 heatmap;
8. enlarge the heatmap to 224 x 224;
9. mark as animal every pixel whose heatmap value is at least the heatmap's mean;
10. resize that mask back to the size of the photo.

Its score is the baseline: your points count from there upward.

In [ ]:
# Step 1: the text embeddings of the breed names, one vector of 512 numbers each
with torch.no_grad():
    tokens = processor(text=class_names, return_tensors="pt", padding=True).to(device)
    classes_encoded = model.text_projection(model.text_model(**tokens).pooler_output)
    classes_encoded = F.normalize(classes_encoded, dim=-1)
print(classes_encoded.shape)


In [ ]:
PATCH_SIZE = 16
CLIP_IMG_SIZE = 224
NUM_PATCHES = CLIP_IMG_SIZE // PATCH_SIZE


def get_clip_representation(imgs):
    """CLIP embeddings of one picture or a list of pictures: one vector of 512 numbers each."""
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
        inputs = processor(images=imgs, return_tensors="pt").to(device)
        outputs = model.vision_model(pixel_values=inputs["pixel_values"])
        images_encoded = model.visual_projection(outputs.pooler_output)
    return F.normalize(images_encoded.float(), dim=-1)


def generate_seg_masks(imgs_path, classes_encoded, img_names=None):
    if img_names is None:
        img_names = sorted(os.listdir(imgs_path))
    imgs_names = []
    heatmaps = []
    for img_name in tqdm(img_names):
        # loading the photo, and its height and width
        img = Image.open(os.path.join(imgs_path, img_name)).convert("RGB")
        img_shapes = np.array(img).shape

        # steps 2 to 4: the embedding of the whole photo, and the breed it is most similar to
        image_encoded = get_clip_representation(img)
        chosen_class_num = (image_encoded @ classes_encoded.T).argmax(axis=1)[0]
        chosen_class_emb = classes_encoded[chosen_class_num]

        # steps 5 and 6: 14 x 14 patches of 16 x 16 pixels, each embedded as a picture of its own
        img = np.array(img.resize((CLIP_IMG_SIZE, CLIP_IMG_SIZE)))
        img_patches = [Image.fromarray(img[x:x + PATCH_SIZE, y:y + PATCH_SIZE])
                       for x in range(0, CLIP_IMG_SIZE, PATCH_SIZE) for y in range(0, CLIP_IMG_SIZE, PATCH_SIZE)]
        img_patches_embs = get_clip_representation(img_patches)

        # steps 7 to 9: the heatmap of similarities, enlarged, scaled to 0..1, cut at its mean
        heatmap = img_patches_embs @ chosen_class_emb.unsqueeze(0).T
        heatmap = heatmap.reshape(NUM_PATCHES, NUM_PATCHES).unsqueeze(0)
        heatmap = F.interpolate(heatmap[:, None], scale_factor=PATCH_SIZE, mode="bilinear")
        heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
        heatmap = heatmap.ge(heatmap.mean()).float()

        # step 10: back to the size of the photo
        heatmap = F.interpolate(heatmap, size=(img_shapes[0], img_shapes[1]), mode="nearest")
        imgs_names.append(img_name)
        heatmaps.append(heatmap[0, 0].cpu().numpy().astype(np.uint8))
    return imgs_names, heatmaps


The IoU of one mask, as the score counts it for every test photo:

In [ ]:
def binaryMaskIOU(mask1, mask2):
    assert mask1.shape == mask2.shape
    mask1_area = np.count_nonzero(mask1 == 1)
    mask2_area = np.count_nonzero(mask2 == 1)
    intersection = np.count_nonzero(np.logical_and(mask1 == 1, mask2 == 1))
    union = mask1_area + mask2_area - intersection
    return intersection / union if union > 0 else 1.0


The starter on the 20 labelled photos, and its mean IoU there:

In [ ]:
train_path = os.path.join(DATA_DIR, "train_images")
train_masks_path = os.path.join(DATA_DIR, "train_masks")

train_names, train_seg_masks = generate_seg_masks(train_path, classes_encoded)
train_ious = []
for img_name, seg_mask in zip(train_names, train_seg_masks):
    mask = np.array(Image.open(os.path.join(train_masks_path, img_name.replace(".jpg", ".png")))) // 255
    train_ious.append(binaryMaskIOU(seg_mask, mask))
print("Mean IoU on the labelled photos:", round(float(np.mean(train_ious)), 4))


One of its masks, beside the photo and the true mask:

In [ ]:
NUM_VIS_IMG = 0
f, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Image.open(os.path.join(train_path, train_names[NUM_VIS_IMG])))
axes[1].imshow(train_seg_masks[NUM_VIS_IMG], cmap="gray")
axes[2].imshow(Image.open(os.path.join(train_masks_path, train_names[NUM_VIS_IMG].replace(".jpg", ".png"))), cmap="gray")
axes[0].set_title("photo")
axes[1].set_title("starter's mask")
axes[2].set_title("true mask")
for ax in axes:
    ax.axis("off")
plt.show()


The function that writes your file from a list of file names and a list of 0/1 masks:

In [ ]:
def rle_encode(mask):
    """A 0/1 mask as pairs "start length" of its 1 pixels: pixels numbered from 1, row by row."""
    pixels = np.concatenate([[0], mask.flatten().astype(np.int8), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


def generate_submit(img_names, seg_masks):
    sub = pd.DataFrame()
    sub["id"] = [int(img_name[:-4]) for img_name in img_names]
    sub["mask"] = [rle_encode(seg_mask) for seg_mask in seg_masks]
    sub = sub.sort_values("id")
    sub.to_csv("submission.csv", index=False)
    print(sub.shape)
    print(sub.head())


The starter on the 1,000 test photos, then its file. It is slow: every photo needs 197 runs of CLIP. On a Colab GPU (Runtime > Change runtime type > T4 GPU) count on 15 to 30 minutes; without a GPU it takes hours.

In [ ]:
test_path = os.path.join(DATA_DIR, "test_images")
test_names, test_seg_masks = generate_seg_masks(test_path, classes_encoded)

generate_submit(test_names, test_seg_masks)
ck.submit("submission.csv")


---
# Hand in

**Every file goes to Kaggle.** `ck.submit` checks the format of `submission.csv` and records it; the score comes from Kaggle. On Colab, download the file first: open the Files panel (the folder icon on the left), find `submission.csv`, click the three dots beside it, then Download. On the competition's Kaggle page click **Submit Prediction**, choose the file and click **Submit**. Do the same, and run `ck.submit("submission.csv")` again, each time you write a better file.

**Your own cells go below**, as many as you need, above the Finish cell.

**Your final notebook** must run from top to bottom in under 20 minutes, and the file it writes last is the one you choose as your final submission on Kaggle. The starter's pass over the test photos is long: once you have a file of your own, delete that cell.

In [ ]:
# Your code here


---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---

Made By **Sattam Altwaim**